# Fine-tune Qwen 2.5 7B Instruct on BrainCode, then run ProofWriter

**Part 1 (phase 1)** fine-tunes Qwen 2.5 7B Instruct with LoRA (Unsloth) on the validated BrainCode translations (`data/train.jsonl`) and saves the adapter to your Drive.

**Part 2 (phases 3 and 4a)** serves the base model and the adapter with vLLM, and runs the 70 ProofWriter items (depth-5 OWA test, proof depth 3 and 4):
- `baseline`: plain Qwen 2.5 on the English items;
- `braincode_ft`: the fine-tuned Qwen on the items' BrainCode (Gemini's successful translations, phase 2). It ends with its conclusion in BrainCode, which Gemini back-translators read back in the repository (phase 4b).

### Before you start
1. Run phase 2 locally first (`evaluations/proofwriter/pw.py translate`, then `package`). `benchmark/sample.jsonl` must hold the translations.
2. Upload this folder to **MyDrive** as `finetune-qwen.zip` (fast) or as the folder `MyDrive/finetune-qwen`.
3. **Runtime → Change runtime type → GPU.** An **A100** is best. An **L4** works. A **T4** works with the 4-bit model.

Run the cells top to bottom. Part 2 starts with a runtime restart.

## Part 1: fine-tune (phase 1)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
import os
os.chdir('/content')     # never work inside the Drive mount: if it drops, every import fails (Errno 107)
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/finetune-qwen'   # results go here on Drive (change if elsewhere)
ZIP = '/content/drive/MyDrive/finetune-qwen.zip'  # the folder as one zip: copies in seconds
WORK = '/content/finetune-qwen'                   # a local copy to work from: fast, and immune to Drive drops
if not os.path.exists(f'{WORK}/data/train.jsonl'):  # after a session restart the local copy is still there
    if os.path.exists(ZIP):
        !unzip -q -o "{ZIP}" -d /content
    else:                                          # file by file from Drive: slow (minutes), shows progress
        assert os.path.exists(f'{FOLDER}/data/train.jsonl'), 'upload finetune-qwen.zip (or the folder) to MyDrive'
        !rsync -a --info=progress2 --exclude runs --exclude outputs --exclude models "{FOLDER}/" "{WORK}/"
OUT = f'{FOLDER}/outputs'                          # adapter and zipped results are saved on Drive
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'{FOLDER}/runs', exist_ok=True)       # benchmark runs are written straight to Drive (resumable)
if not os.path.islink(f'{WORK}/runs'):
    os.symlink(f'{FOLDER}/runs', f'{WORK}/runs')
%cd {WORK}

Mounted at /content/drive
     10,741,088 100%   18.32kB/s    0:09:32 (xfr#599, to-chk=0/623)
/content/finetune-qwen


In [ ]:
%%capture
!pip install unsloth

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # less fragmentation (set before torch loads)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'   # plain downloads: the accelerated paths can truncate tokenizer.json
os.environ['HF_HUB_DISABLE_XET'] = '1'
import torch

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
LOAD_IN_4BIT = GPU_GB < 40          # A100: 16-bit LoRA; L4/T4: 4-bit QLoRA
BASE_MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' if LOAD_IN_4BIT else 'unsloth/Qwen2.5-7B-Instruct'
MAX_SEQ_LEN = 8192 if GPU_GB >= 40 else 4096
READ_DIRECTION = True   # also train BrainCode -> English (each translation read back). Phase 4 gives the model
                        # BrainCode to read; False trains only English -> BrainCode, as in the Llama run.
EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_R = 16
BATCH, GRAD_ACC = (2, 4) if GPU_GB >= 70 else (1, 8)
ADAPTER_DIR = f'{OUT}/braincode-qwen-lora'
print(f'GPU {GPU_GB:.0f} GB -> {BASE_MODEL}, max length {MAX_SEQ_LEN}, batch {BATCH} x {GRAD_ACC}')

GPU 42 GB -> unsloth/Qwen2.5-7B-Instruct, max length 8192, batch 1 x 8


### Training data
Each translation gives a **write** example: the task in English, answered with its BrainCode. With `READ_DIRECTION`, it also gives a **read** example: the BrainCode, answered with the task in English. The loss is on the answers only.

In [ ]:
import json, random

WRITE_SYSTEM = 'You translate tasks into BrainCode, a formal language with a fixed glossary of symbols. You do not solve the task.'
READ_SYSTEM = 'You read and write BrainCode, a formal language with a fixed glossary of symbols.'

def to_chats(row):
    fenced = '```braincode\n' + row['braincode'] + '\n```'
    chats = [[{'role': 'system', 'content': WRITE_SYSTEM},
              {'role': 'user', 'content': 'Translate this task into BrainCode:\n\n' + row['item']},
              {'role': 'assistant', 'content': fenced}]]
    if READ_DIRECTION:
        chats.append([{'role': 'system', 'content': READ_SYSTEM},
                      {'role': 'user', 'content': 'Write this BrainCode task in plain English:\n\n' + fenced},
                      {'role': 'assistant', 'content': row['item']}])
    return chats

def load(name):
    with open(f'{WORK}/data/{name}.jsonl', encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

train_rows, val_rows = load('train'), load('val')
print(json.load(open(f'{WORK}/data/manifest.json'))['by_dataset'])
print(len(train_rows), 'train /', len(val_rows), 'validation translations;',
      'write + read examples' if READ_DIRECTION else 'write examples only')

{'alfred': 83, 'mind2web': 110, 'paths': 95, 'prism': 105, 'swebench': 68, 'thoughttrace': 105}
535 train / 31 validation translations; write + read examples


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

def load_base():
    return FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_IN_4BIT, dtype=None)

free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb > (8 if LOAD_IN_4BIT else 20), (
    f'only {free_gb:.1f} GB of GPU memory is free: an earlier model is still loaded. '
    'Runtime -> Restart session, then run the cells again from the Drive-mount cell (skip pip install).')
try:
    model, tokenizer = load_base()
except Exception as e:
    if 'expected value' not in str(e) and 'tokenizer' not in str(e).lower():
        raise
    # a truncated download ("expected value at line 1 column 1" from tokenizer.json): clear it and retry once
    import glob, shutil
    print('load failed:', e, '\n-> clearing the cached download and retrying once')
    for d in glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--' + BASE_MODEL.replace('/', '--') + '*')):
        shutil.rmtree(d, ignore_errors=True)
    model, tokenizer = load_base()
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=2026)
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.1: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.10.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import Dataset

def as_text(rows):
    texts = [tokenizer.apply_chat_template(c, tokenize=False) for r in rows for c in to_chats(r)]
    lengths = [len(tokenizer(t).input_ids) for t in texts]
    kept = [t for t, n in zip(texts, lengths) if n <= MAX_SEQ_LEN]
    print(f'{len(kept)}/{len(texts)} examples fit in {MAX_SEQ_LEN} tokens (longest {max(lengths)})')
    random.Random(2026).shuffle(kept)
    return Dataset.from_dict({'text': kept})

train_ds, val_ds = as_text(train_rows), as_text(val_rows)

1060/1070 examples fit in 8192 tokens (longest 17771)
62/62 examples fit in 8192 tokens (longest 4989)


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, lr_scheduler_type='cosine', warmup_steps=10,
        weight_decay=0.01, optim='adamw_8bit', logging_steps=5, eval_strategy='epoch', save_strategy='no',
        bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
        output_dir='/content/checkpoints', report_to='none', seed=2026))
# learn only the answers, not the prompts
trainer = train_on_responses_only(trainer,
    instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(stats)
print(trainer.evaluate())

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1060 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/62 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/1060 [00:00<?, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss
1,0.596171,0.470188
2,0.467625,0.479645


Filter:   0%|          | 0/62 [00:00<?, ? examples/s]

TrainOutput(global_step=266, training_loss=0.5612415478641826, metrics={'train_runtime': 1079.8569, 'train_samples_per_second': 1.963, 'train_steps_per_second': 0.246, 'total_flos': 1.4711726679896064e+17, 'train_loss': 0.5612415478641826, 'epoch': 2.0})


Training Loss,Validation Loss,Epoch
0.467625,0.479645,2


{'eval_loss': 0.4796450138092041}


In [ ]:
# Save the LoRA adapter to Drive (~160 MB). Part 2 serves it with vLLM.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('saved', ADAPTER_DIR)

# Optional: push the adapter to the Hugging Face Hub (Colab secret HF_TOKEN, write permission)
PUSH_TO_HUB = False
HUB_REPO = 'LiaSoffer/braincode-qwen2.5-7b-lora'
if PUSH_TO_HUB:
    from google.colab import userdata
    model.push_to_hub(HUB_REPO, token=userdata.get('HF_TOKEN'), private=True)
    tokenizer.push_to_hub(HUB_REPO, token=userdata.get('HF_TOKEN'), private=True)

saved /content/drive/MyDrive/finetune-qwen/outputs/braincode-qwen-lora


### Sanity check: translate a held-out item and run the BrainCode checker on it

In [ ]:
import sys
sys.path.insert(0, f'{WORK}/braincode_kit')
!pip -q install rank_bm25
from rag.retrieve import Retriever, render_check

FastLanguageModel.for_inference(model)
row = val_rows[0]
msgs = to_chats(row)[0][:2]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=2048, temperature=0.7, top_p=0.8, do_sample=True)
answer = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(answer[:3000])

checker = Retriever(dense=False, glossary_path=f'{WORK}/braincode_kit/reference/glossary.jsonl',
                    index_dir=f'{WORK}/braincode_kit/rag/index', use_llm=False)
print(render_check(checker.check(answer, [])))

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```braincode
MODE TRACE
ENTRYPOINT Conversation
CONVO Conversation {
  TURN t1 SPEAKER=USER {
    TERM location_spec(city="Belo Horizonte") -> location_spec_2 : TERM
    CLAIM request(target=location_spec_2) BY role_user STATUS asserted SOURCE "t1:s1" -> request_2 : CLAIM
  }
  TURN t2 SPEAKER=AGENT {
    TERM web_element(label="Search", tag="textbox") -> web_element_2 : TERM
    RECORD ACTION type_text(target=web_element_2, text="Belo Horizonte") STATUS succeeded SOURCE "t2:s2" -> type_text_event : EVENT
    TERM web_element(label="Belo Horizonte, Minas Gerais, BR", tag="div") -> web_element_3 : TERM
    RECORD ACTION click(target=web_element_3) STATUS succeeded SOURCE "t2:s4" -> click_event : EVENT
    TERM web_element(label="", tag="use") -> web_element_4 : TERM
    RECORD ACTION click(target=web_element_4) STATUS succeeded SOURCE "t2:s6" -> click_event_2 : EVENT
    TERM web_element(label="Radar & Maps", tag="heading") -> web_element_5 : TERM
    RECORD ACTION click(target=web_elem

## Part 2: serve the models and run ProofWriter (phases 3 and 4a)

**First: Runtime → Restart session.** This frees the GPU memory from training and gives vLLM a clean environment. Files on Drive and in `/content` stay. Python variables are lost, so the next cell sets the paths again.

If your session disconnected after Part 1, start here: the adapter is on Drive.

In [1]:
import os
os.chdir('/content')     # never work inside the Drive mount: if it drops, every import fails (Errno 107)
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/finetune-qwen'   # results go here on Drive (change if elsewhere)
ZIP = '/content/drive/MyDrive/finetune-qwen.zip'  # the folder as one zip: copies in seconds
WORK = '/content/finetune-qwen'                   # a local copy to work from: fast, and immune to Drive drops
if not os.path.exists(f'{WORK}/data/train.jsonl'):  # after a session restart the local copy is still there
    if os.path.exists(ZIP):
        !unzip -q -o "{ZIP}" -d /content
    else:                                          # file by file from Drive: slow (minutes), shows progress
        assert os.path.exists(f'{FOLDER}/data/train.jsonl'), 'upload finetune-qwen.zip (or the folder) to MyDrive'
        !rsync -a --info=progress2 --exclude runs --exclude outputs --exclude models "{FOLDER}/" "{WORK}/"
OUT = f'{FOLDER}/outputs'                          # adapter and zipped results are saved on Drive
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'{FOLDER}/runs', exist_ok=True)       # benchmark runs are written straight to Drive (resumable)
if not os.path.islink(f'{WORK}/runs'):
    os.symlink(f'{FOLDER}/runs', f'{WORK}/runs')
%cd {WORK}

import torch, json, shutil
ADAPTER_DIR = f'{FOLDER}/outputs/braincode-qwen-lora'
assert os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'), 'no adapter: run Part 1 first'
if os.path.exists(f'{FOLDER}/benchmark/sample.jsonl'):   # a newer benchmark/ uploaded to Drive (after phase 2) wins
    shutil.copytree(f'{FOLDER}/benchmark', f'{WORK}/benchmark', dirs_exist_ok=True)
items = [json.loads(l) for l in open(f'{WORK}/benchmark/sample.jsonl', encoding='utf-8')]
print(len(items), 'items,', sum(bool(i['braincode']) for i in items), 'with a successful BrainCode translation')
if not any(i['braincode'] for i in items):
    print('No translations in benchmark/ yet: the baseline can run now. For phase 4, run `pw.py package` locally, '
          f'upload finetune-qwen/benchmark/ to {FOLDER}/benchmark, and rerun this cell.')
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
if GPU_GB >= 20:      # A100 / L4: 16-bit weights
    SERVE_MODEL, EXTRA, MAX_MODEL_LEN = 'Qwen/Qwen2.5-7B-Instruct', '', 32768 if GPU_GB >= 40 else 20480
else:                 # T4: 16-bit weights do not fit; serve the 4-bit model (fp16, no bf16 on T4)
    SERVE_MODEL, EXTRA, MAX_MODEL_LEN = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit', '--quantization bitsandbytes --dtype half', 16384
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
print(f'GPU {GPU_GB:.0f} GB -> serving {SERVE_MODEL} with {MAX_MODEL_LEN} tokens of context')

if os.path.exists(f'{FOLDER}/run_proofwriter.py'):       # a newer runner uploaded to Drive wins
    shutil.copy2(f'{FOLDER}/run_proofwriter.py', f'{WORK}/run_proofwriter.py')


Mounted at /content/drive
     11,955,173 100%   22.77kB/s    0:08:32 (xfr#600, to-chk=0/624)
/content/finetune-qwen
70 items, 67 with a successful BrainCode translation
GPU 42 GB -> serving Qwen/Qwen2.5-7B-Instruct with 32768 tokens of context


In [7]:
import shutil, json
shutil.copytree(f'{FOLDER}/benchmark', f'{WORK}/benchmark', dirs_exist_ok=True)
shutil.copy2(f'{FOLDER}/run_proofwriter.py', f'{WORK}/run_proofwriter.py')   # the newest runner on Drive wins
items = [json.loads(l) for l in open(f'{WORK}/benchmark/sample.jsonl', encoding='utf-8')]
print(len(items), 'items,', sum(bool(i['braincode']) for i in items), 'with a translation')
assert 'VERDICT_PROMPT' in open(f'{WORK}/run_proofwriter.py').read(), (
    'old run_proofwriter.py: upload the two-call version to MyDrive/finetune-qwen/ and rerun this cell')
print('two-call runner OK')

70 items, 67 with a translation
two-call runner OK


### One time: set aside the one-call BrainCode runs
The first BrainCode runs (`smoke_braincode_ft`, `braincode_ft`, `braincode_base`) used the one-call runner: the model mostly re-translated the problem instead of solving it. This cell renames those folders to `*_onecall` (kept for the record; the analysis ignores them), so the two-call runner starts fresh. Runs that already have `reasoning.md` (two-call) are left alone. The English runs (`baseline`, `baseline_ft`) are valid and stay.

In [8]:
import glob, os
for c in ('smoke_braincode_ft', 'braincode_ft', 'braincode_base'):
    d = f'{FOLDER}/runs/{c}'
    if os.path.isdir(d) and not glob.glob(f'{d}/*/r*/reasoning.md'):
        dest = d + '_onecall'
        assert not os.path.exists(dest), f'{dest} exists already: move one of them away by hand'
        os.rename(d, dest)
        print('set aside', c, '->', c + '_onecall')
print('BrainCode runs now:', sorted(os.path.basename(p) for p in glob.glob(f'{FOLDER}/runs/*')))

BrainCode runs now: ['baseline', 'baseline_ft', 'smoke_baseline', 'smoke_braincode_ft', 'smoke_braincode_ft_onecall']


In [4]:
%%capture
!pip install vllm

In [9]:
# vLLM serves the base model as `base-qwen` and the fine-tuned adapter on top of it as `braincode-qwen`.
import subprocess, time, urllib.request
cmd = (f'vllm serve {SERVE_MODEL} --served-model-name base-qwen {EXTRA} '
       f'--enable-lora --lora-modules braincode-qwen={ADAPTER_DIR} --max-lora-rank 16 '
       f'--max-model-len {MAX_MODEL_LEN} --gpu-memory-utilization 0.90 --port 8000')
server = subprocess.Popen(cmd, shell=True, stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(180):                       # model download + load: usually 3-8 minutes
    try:
        models = json.loads(urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5).read())
        print('server up:', [m['id'] for m in models['data']])
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError('vLLM exited; see /content/vllm.log:\n' + open('/content/vllm.log').read()[-3000:])
        time.sleep(10)
else:
    print(open('/content/vllm.log').read()[-3000:])

server up: ['base-qwen', 'braincode-qwen']


### Smoke test: 2 items, 1 run per condition (read the responses before the full run)

In [11]:
import shutil, os
src, dst = f'{FOLDER}/run_proofwriter.py', f'{WORK}/run_proofwriter.py'
print('Drive runner is new:', 'VERDICT_PREFILL' in open(src).read())
shutil.copy2(src, dst)
print('Working runner is new:', 'VERDICT_PREFILL' in open(dst).read())
smoke = f'{FOLDER}/runs/smoke_braincode_ft'
shutil.rmtree(smoke, ignore_errors=True)
print('Old smoke runs cleared:', not os.path.exists(smoke))


Drive runner is new: True
Working runner is new: True
Old smoke runs cleared: True


In [12]:
!python run_proofwriter.py --model base-qwen --condition smoke_baseline --input english --runs 1 --limit 2
!python run_proofwriter.py --model braincode-qwen --condition smoke_braincode_ft --input braincode --runs 1 --limit 2
import glob
for run in sorted(glob.glob(f'{WORK}/runs/smoke_braincode_ft/*/r1')):
    print('=' * 30, run.split('/')[-2])
    print('--- reasoning (call 1) ---\n' + open(f'{run}/reasoning.md').read()[-3000:])
    print('--- verdict (call 2) ---\n' + open(f'{run}/response.md').read())

smoke_baseline: 2 runs (2 items x 1) on base-qwen, english input
67 items with a successful translation (3 without one are left out)
smoke_braincode_ft: 2 runs (2 items x 1) on braincode-qwen, braincode input
smoke_braincode_ft proofwriter-d3-01 r1 -> verdict block; English answer unknown (gold True)
smoke_braincode_ft proofwriter-d3-02 r1 -> verdict block; English answer unknown (gold True)
============================== proofwriter-d3-01
--- reasoning (call 1) ---
The answer is: Unknown

Explanation:
1. The problem provides various statements about the relationships and characteristics of animals (cow, lion, rabbit, and squirrel) in a zoo setting.
2. We are asked if the rabbit is round based only on the provided theory.
3. While the theory provides some information about the animals, it does not explicitly mention the shape or roundness of the rabbit.
4. Therefore, we cannot definitively determine if the rabbit is round or not based solely on the given theory.
5. Since the informatio

**Check before the full run:** the reasoning should derive facts with the rules (not restate or re-encode the problem), and the verdict should be a short block: the statement as `q`, then one of the three verdict forms. If not, stop and send the output.

### Phase 3: the baseline, plain Qwen 2.5 on the English items (70 × 3)
Resumable: if Colab disconnects, rerun the cell. Finished runs are kept and errored runs are redone.

In [ ]:
!python run_proofwriter.py --model base-qwen --condition baseline --input english --runs 3

baseline: 210 runs (70 items x 3) on base-qwen, english input
baseline proofwriter-d3-04 r1 -> wrong (pred unknown, gold True)
baseline proofwriter-d3-01 r1 -> wrong (pred unknown, gold True)
baseline proofwriter-d3-06 r1 -> wrong (pred unknown, gold True)
baseline proofwriter-d3-13 r1 -> correct (pred false, gold False)
baseline proofwriter-d3-15 r1 -> wrong (pred unknown, gold False)
baseline proofwriter-d3-02 r1 -> correct (pred true, gold True)
baseline proofwriter-d3-11 r1 -> wrong (pred unknown, gold True)
baseline proofwriter-d3-03 r1 -> correct (pred true, gold True)
baseline proofwriter-d3-14 r1 -> wrong (pred unknown, gold False)
baseline proofwriter-d3-09 r1 -> wrong (pred unknown, gold True)
baseline proofwriter-d3-18 r1 -> correct (pred false, gold False)
baseline proofwriter-d3-07 r1 -> wrong (pred unknown, gold True)
baseline proofwriter-d3-05 r1 -> correct (pred true, gold True)
baseline proofwriter-d3-20 r1 -> correct (pred false, gold False)
baseline proofwriter-d3-08

### Phase 4a: the fine-tuned Qwen on the successful BrainCode translations (67 × 3)
Two calls per run, in one conversation: the model reasons in English from the BrainCode problem (`reasoning.md`), then writes only its verdict as a BrainCode block (`response.md`). Back in the repository, Gemini back-translators read that block (phase 4b). Resumable.

In [13]:
!python run_proofwriter.py --model braincode-qwen --condition braincode_ft --input braincode --runs 3

67 items with a successful translation (3 without one are left out)
braincode_ft: 201 runs (67 items x 3) on braincode-qwen, braincode input
braincode_ft proofwriter-d3-16 r1 -> verdict block; English answer none (gold False)
braincode_ft proofwriter-d3-04 r1 -> verdict block; English answer unknown (gold True)
braincode_ft proofwriter-d3-09 r1 -> verdict block; English answer unknown (gold True)
braincode_ft proofwriter-d3-02 r1 -> verdict block; English answer unknown (gold True)
braincode_ft proofwriter-d3-10 r1 -> verdict block; English answer unknown (gold True)
braincode_ft proofwriter-d3-11 r1 -> verdict block; English answer unknown (gold True)
braincode_ft proofwriter-d3-08 r1 -> verdict block; English answer unknown (gold True)
braincode_ft proofwriter-d3-01 r1 -> verdict block; English answer unknown (gold True)
braincode_ft proofwriter-d3-15 r1 -> verdict block; English answer none (gold False)
braincode_ft proofwriter-d3-13 r1 -> verdict block; English answer unknown (gold

### Optional controls
- `braincode_base`: the plain model on the same BrainCode. Is any gain from fine-tuning, or from the representation alone?
- `baseline_ft`: the fine-tuned model on English. Did fine-tuning change plain reasoning?

In [14]:
!python run_proofwriter.py --model base-qwen --condition braincode_base --input braincode --runs 3
!python run_proofwriter.py --model braincode-qwen --condition baseline_ft --input english --runs 3

67 items with a successful translation (3 without one are left out)
braincode_base: 201 runs (67 items x 3) on base-qwen, braincode input
braincode_base proofwriter-d3-05 r1 -> verdict block; English answer true (gold True)
braincode_base proofwriter-d3-03 r1 -> verdict block; English answer true (gold True)
braincode_base proofwriter-d3-06 r1 -> verdict block; English answer true (gold True)
braincode_base proofwriter-d3-04 r1 -> verdict block; English answer false (gold True)
braincode_base proofwriter-d3-15 r1 -> verdict block; English answer unknown (gold False)
braincode_base proofwriter-d3-01 r1 -> verdict block; English answer true (gold True)
braincode_base proofwriter-d3-07 r1 -> verdict block; English answer true (gold True)
braincode_base proofwriter-d3-02 r1 -> verdict block; English answer true (gold True)
braincode_base proofwriter-d3-08 r1 -> verdict block; English answer none (gold True)
braincode_base proofwriter-d3-14 r1 -> verdict block; English answer false (gold Fa

### Progress, and a zip of all runs to bring back to the repository

In [15]:
import glob, shutil
from collections import Counter
c = Counter()
for p in glob.glob(f'{FOLDER}/runs/*/*/r*/result.json'):
    r = json.load(open(p))
    c[(r['condition'], 'error' if r['status'] != 'ok' else 'awaiting back-translation' if r['correct'] is None
       else 'correct' if r['correct'] else 'wrong')] += 1
for (cond, state), n in sorted(c.items()):
    print(f'{cond:22} {state:26} {n}')
shutil.make_archive(f'{FOLDER}/outputs/runs', 'zip', f'{FOLDER}/runs')
print('zipped ->', f'{FOLDER}/outputs/runs.zip')

baseline               correct                    115
baseline               wrong                      95
baseline_ft            correct                    90
baseline_ft            wrong                      120
braincode_base         awaiting back-translation  201
braincode_ft           awaiting back-translation  201
smoke_baseline         correct                    1
smoke_baseline         wrong                      1
smoke_braincode_ft     awaiting back-translation  4
zipped -> /content/drive/MyDrive/finetune-qwen/outputs/runs.zip
